# NB4 — Evaluation, Analysis & Final Export
**Person 4 runs this AFTER NB3 is complete.**

## Setup before running:
1. In Kaggle → `+ Add Data` → search for `plutchik-model-v2` (from Person 3) → add it
2. GPU not required (but fine to enable)
3. Run all cells


## 0. Install

In [ ]:
%%capture
!pip install transformers accelerate datasets scipy scikit-learn matplotlib -q
print('done')

## 1. Config & Load Model

In [ ]:
import os, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from scipy.stats import spearmanr, pearsonr

MODEL_DIR = '/kaggle/input/plutchik-model-v2'
EMOTIONS  = ['joy','trust','fear','surprise','sadness','disgust','anger','anticipation']
N_EMO     = len(EMOTIONS)
MAX_LEN   = 128
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 2. Rebuild Model Architecture (must match NB3 exactly)

In [ ]:
class EmotionAttentionBlock(nn.Module):
    def __init__(self, hidden_size, n_heads=4, out_dim=128):
        super().__init__()
        self.n_heads  = n_heads
        self.head_dim = hidden_size // n_heads
        self.scale    = self.head_dim ** -0.5
        self.query    = nn.Parameter(torch.randn(1, 1, hidden_size) * 0.02)
        self.k_proj   = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v_proj   = nn.Linear(hidden_size, hidden_size, bias=False)
        self.out_proj = nn.Linear(hidden_size, out_dim)
        self.norm     = nn.LayerNorm(out_dim)
        self.dropout  = nn.Dropout(0.1)

    def forward(self, hidden_states, attention_mask):
        B, L, H = hidden_states.shape
        nh, hd  = self.n_heads, self.head_dim
        q = self.query.expand(B, -1, -1)
        k = self.k_proj(hidden_states)
        v = self.v_proj(hidden_states)
        q = q.view(B, 1, nh, hd).transpose(1, 2)
        k = k.view(B, L, nh, hd).transpose(1, 2)
        v = v.view(B, L, nh, hd).transpose(1, 2)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        if attention_mask is not None:
            pad = (attention_mask == 0).unsqueeze(1).unsqueeze(2)
            attn = attn.masked_fill(pad, float('-inf'))
        attn = self.dropout(torch.softmax(attn, dim=-1))
        out  = (attn @ v).squeeze(2).reshape(B, H)
        return self.norm(self.out_proj(out))


class PlutchikModelV2(nn.Module):
    def __init__(self, base_model_name, n_emotions=N_EMO, out_dim=128, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model_name)
        H = self.encoder.config.hidden_size
        self.emotion_blocks = nn.ModuleList([EmotionAttentionBlock(H, 4, out_dim) for _ in range(n_emotions)])
        self.emotion_heads  = nn.ModuleList([
            nn.Sequential(nn.Linear(out_dim, 32), nn.GELU(), nn.Dropout(dropout), nn.Linear(32, 1))
            for _ in range(n_emotions)
        ])
        self.temperature = nn.Parameter(torch.full((n_emotions,), 2.0))
        self.conf_head   = nn.Sequential(nn.Linear(H, 128), nn.GELU(), nn.Dropout(dropout), nn.Linear(128, 1), nn.Sigmoid())
        # Aux classification head (must match NB3 for state_dict loading)
        self.cls_head    = nn.Sequential(nn.Linear(H, 128), nn.GELU(), nn.Dropout(dropout), nn.Linear(128, n_emotions))
        self.drop = nn.Dropout(dropout)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        enc  = self.encoder(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        h    = self.drop(enc.last_hidden_state)
        cls  = h[:, 0]
        logits = [head(block(h, attention_mask)) for block, head in zip(self.emotion_blocks, self.emotion_heads)]
        logits = torch.cat(logits, dim=-1)
        temp   = torch.clamp(self.temperature, 0.5, 5.0)
        scores = torch.sigmoid(logits * temp)
        return scores, self.conf_head(cls).squeeze(-1), self.cls_head(cls)


# Load config to get base model name
with open(os.path.join(MODEL_DIR, 'model_config.json')) as f:
    cfg = json.load(f)

print(f'Architecture: {cfg["architecture"]}')
print('Loading model...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = PlutchikModelV2(cfg['base_model']).to(DEVICE)
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_model.pt'), map_location=DEVICE))
model.eval()
print(f'Model loaded. Best val Spearman: {cfg["best_val_spearman"]:.4f}')

## 3. Predict Function

In [ ]:
def predict(texts):
    if isinstance(texts, str):
        texts = [texts]
    enc = tokenizer(
        texts, max_length=MAX_LEN, padding='max_length',
        truncation=True, return_tensors='pt'
    ).to(DEVICE)
    with torch.no_grad():
        scores, conf, _ = model(
            enc['input_ids'], enc['attention_mask'],
            enc.get('token_type_ids'),
        )
    scores = scores.float().cpu().numpy()
    conf   = conf.float().cpu().numpy()
    results = []
    for i, text in enumerate(texts):
        row = {e: float(scores[i, j]) for j, e in enumerate(EMOTIONS)}
        row['confidence'] = float(conf[i])
        results.append(row)
    return results if len(results) > 1 else results[0]


print('predict() ready')

## 4. Qualitative Analysis — 20+ Test Sentences

In [ ]:
TEST_SENTENCES = [
    # Multi-emotion (complex)
    ('multi',   'She was absolutely thrilled to see him but terrified of what came next.'),
    ('multi',   'I love this city even though it constantly disappoints me.'),
    ('multi',   'The news was unexpected and left everyone in shock, though some felt relieved.'),
    ('multi',   'It was the best day of his life, and also the worst.'),
    # Single strong emotion
    ('anger',   "I can't believe they did that. It's disgusting and makes me furious."),
    ('joy',     'This is the happiest moment of my entire life!'),
    ('sadness', 'She cried for hours after hearing the news. Nothing felt real anymore.'),
    ('fear',    'He could hear footsteps behind him. His heart was pounding.'),
    ('trust',   'Everything is going to be okay. I trust that things will work out.'),
    ('disgust', 'The smell was unbearable. How could anyone live like this?'),
    ('surprise','She opened the door and gasped. She could not believe her eyes.'),
    ('anticip', 'The match starts in an hour. I have been waiting for this all week.'),
    # Ambiguous / low signal
    ('ambig',   'I don\'t know what to feel anymore.'),
    ('ambig',   'Meh.'),
    ('ambig',   'When she finally...'),
    ('neutral',  'The train arrives at 9am.'),
    # Sarcasm / irony
    ('irony',   'Oh great, another Monday. Just what I needed.'),
    ('irony',   'Sure, because everything always goes perfectly according to plan.'),
    # Subtle / nuanced
    ('subtle',  'He smiled, even though he knew it was the last time.'),
    ('subtle',  'She kept the letter but never opened it.'),
    ('subtle',  'They laughed until it stopped being funny.'),
]

print(f'{'Category':<10} {'Sentence':<58} | Dominant emotions\n' + '-'*100)
for category, sent in TEST_SENTENCES:
    r = predict(sent)
    top2 = sorted([(e, r[e]) for e in EMOTIONS], key=lambda x: -x[1])[:2]
    dom = '  '.join(f'{e}={v:.3f}' for e,v in top2)
    bar = chr(9608) * int(r[top2[0][1]] * 20)
    print(f'{category:<10} {sent[:57]:<58} | {dom}  {bar}')

## 5. Radar Plot — Selected Sentences

In [ ]:
def plot_radar(texts, figsize=(16, 4)):
    n = len(texts)
    fig, axes = plt.subplots(1, n, figsize=figsize, subplot_kw={'projection': 'polar'})
    if n == 1:
        axes = [axes]

    angles = np.linspace(0, 2*np.pi, N_EMO, endpoint=False).tolist()
    angles += angles[:1]

    for ax, text in zip(axes, texts):
        r = predict(text)
        vals = [r[e] for e in EMOTIONS] + [r[EMOTIONS[0]]]

        ax.plot(angles, vals, 'o-', linewidth=1.5, color='steelblue')
        ax.fill(angles, vals, alpha=0.25, color='steelblue')
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(EMOTIONS, size=8)
        ax.set_ylim(0, 1)
        ax.set_title(text[:35] + ('...' if len(text) > 35 else ''), size=8, pad=10)

    plt.tight_layout()
    plt.savefig('/kaggle/working/radar_plots.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_radar([
    'She was absolutely thrilled to see him but terrified of what came next.',
    "I can't believe they did that. It's disgusting and makes me furious.",
    'He smiled, even though he knew it was the last time.',
    'This is the happiest moment of my entire life!',
])

## 6. Full Test-Set Evaluation

In [ ]:
from torch.utils.data import Dataset, DataLoader


class SimpleDataset(Dataset):
    def __init__(self, df):
        self.texts  = df['text'].tolist()
        self.labels = df[EMOTIONS].values.astype(np.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(self.texts[idx], max_length=MAX_LEN, padding='max_length',
                        truncation=True, return_tensors='pt')
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'token_type_ids': enc.get('token_type_ids', torch.zeros(MAX_LEN, dtype=torch.long)).squeeze(0) if 'token_type_ids' in enc else torch.zeros(MAX_LEN, dtype=torch.long),
            'labels':         torch.tensor(self.labels[idx]),
        }


# Load GoEmotions test split as eval set
print('Loading GoEmotions test split for evaluation...')
GO2PLU = {
    'admiration':'trust','amusement':'joy','anger':'anger','annoyance':'anger',
    'approval':'trust','caring':'trust','confusion':'surprise','curiosity':'anticipation',
    'desire':'anticipation','disappointment':'sadness','disapproval':'disgust',
    'disgust':'disgust','embarrassment':'fear','excitement':'anticipation',
    'fear':'fear','gratitude':'joy','grief':'sadness','joy':'joy','love':'joy',
    'nervousness':'fear','optimism':'anticipation','pride':'joy','realization':'surprise',
    'relief':'joy','remorse':'sadness','sadness':'sadness','surprise':'surprise', 'neutral':'joy',
}
EPS = 0.05
go_test = load_dataset('go_emotions', 'simplified', split='test')
label_names = go_test.features['labels'].feature.names
rows = []
for row in go_test:
    vec = {e: EPS for e in EMOTIONS}
    for lid in (row['labels'] or []):
        pe = GO2PLU.get(label_names[lid])
        if pe:
            vec[pe] = min(1.0, vec[pe] + (1.0 - EPS))
    maxv = max(vec.values())
    if maxv > 0:
        vec = {k: v/maxv for k,v in vec.items()}
    rows.append({'text': row['text'], **vec})
eval_df = pd.DataFrame(rows)

eval_ds = SimpleDataset(eval_df)
eval_loader = DataLoader(eval_ds, batch_size=32, num_workers=2, pin_memory=True)

# Evaluate
all_preds, all_targets = [], []
model.eval()
with torch.no_grad():
    for batch in eval_loader:
        ids   = batch['input_ids'].to(DEVICE)
        mask  = batch['attention_mask'].to(DEVICE)
        ttids = batch['token_type_ids'].to(DEVICE)
        scores, _, _ = model(ids, mask, ttids)
        all_preds.append(scores.float().cpu().numpy())
        all_targets.append(batch['labels'].numpy())

preds   = np.concatenate(all_preds)
targets = np.concatenate(all_targets)

print(f'\nEval on {len(eval_df):,} GoEmotions test sentences:')
spearmans = []
for i, e in enumerate(EMOTIONS):
    rho = spearmanr(preds[:,i], targets[:,i])[0] if preds[:,i].std() > 1e-6 else 0.0
    spearmans.append(rho)
    bar = chr(9608) * int(max(0,rho)*20)
    print(f'  {e:<15}: {rho:+.4f}  {bar}')
print(f'  {"Mean Spearman":<15}: {np.mean(spearmans):.4f}')

pn = preds   / (np.linalg.norm(preds,   axis=1, keepdims=True) + 1e-8)
tn = targets / (np.linalg.norm(targets, axis=1, keepdims=True) + 1e-8)
print(f'  {"Vector Cosine":<15}: {(pn*tn).sum(axis=1).mean():.4f}')
print(f'  {"Dom Emo Acc":<15}: {(preds.argmax(1)==targets.argmax(1)).mean():.4f}')

## 7. Score Distribution Analysis (Checks for Compression)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, e in zip(axes.flatten(), EMOTIONS):
    col_idx = EMOTIONS.index(e)
    ax.hist(preds[:, col_idx], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(f'{e.capitalize()}\nmean={preds[:,col_idx].mean():.3f}', fontsize=10)
    ax.set_xlim(0, 1)
    ax.set_ylabel('Count')
plt.suptitle('Predicted Emotion Score Distributions (check for compression near 0)', fontsize=12)
plt.tight_layout()
plt.savefig('/kaggle/working/score_distributions.png', dpi=120)
plt.show()

print('Mean predicted scores per emotion:')
for e in EMOTIONS:
    idx = EMOTIONS.index(e)
    print(f'  {e:<15}: mean={preds[:,idx].mean():.4f}  std={preds[:,idx].std():.4f}  max={preds[:,idx].max():.4f}')

## 8. Training History Plot

In [ ]:
hist_path = os.path.join(MODEL_DIR, 'training_history.csv')
if os.path.exists(hist_path):
    hist = pd.read_csv(hist_path)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(hist['mean_spearman'], 'o-', color='steelblue')
    axes[0].set_title('Mean Spearman (val)'); axes[0].set_ylabel('ρ')
    axes[1].plot(hist['vector_cosine'], 'o-', color='coral')
    axes[1].set_title('Vector Cosine (val)')
    axes[2].plot(hist['dom_acc'], 'o-', color='seagreen')
    axes[2].set_title('Dominant Emotion Acc (val)')
    # Add vertical lines for curriculum stage boundaries
    if 'stage' in hist.columns:
        stages = hist['stage'].values
        for i in range(1, len(stages)):
            if stages[i] != stages[i-1]:
                for ax in axes:
                    ax.axvline(i, color='gray', linestyle='--', alpha=0.5)
    for ax in axes:
        ax.grid(alpha=0.3)
    plt.suptitle('Training History', fontsize=13)
    plt.tight_layout()
    plt.savefig('/kaggle/working/training_history.png', dpi=120)
    plt.show()
else:
    print('No training history CSV found.')

## 9. Error Analysis — 10 Worst Predictions

In [ ]:
cosine_per_sample = (pn * tn).sum(axis=1)
worst_idx = np.argsort(cosine_per_sample)[:10]

print('10 worst predictions (by vector cosine similarity):\n')
for rank, idx in enumerate(worst_idx):
    text = eval_df.iloc[idx]['text']
    pred_top = sorted(enumerate(preds[idx]), key=lambda x: -x[1])[:2]
    true_top = sorted(enumerate(targets[idx]), key=lambda x: -x[1])[:2]
    pred_str = '  '.join(f'{EMOTIONS[i]}={v:.3f}' for i,v in pred_top)
    true_str = '  '.join(f'{EMOTIONS[i]}={v:.3f}' for i,v in true_top)
    print(f'{rank+1}. [{cosine_per_sample[idx]:.3f}] {text[:70]}')
    print(f'   Pred: {pred_str}')
    print(f'   True: {true_str}')
    print()

## 10. Final Summary

In [ ]:
print('='*60)
print('FINAL MODEL SUMMARY')
print('='*60)
print(f'Architecture : {cfg["architecture"]}')
print(f'Base model   : {cfg["base_model"]}')
print(f'Val Spearman : {cfg["best_val_spearman"]:.4f}')
print(f'Test Spearman: {cfg.get("test_spearman", "N/A")}')
print(f'Test Cosine  : {cfg.get("test_cosine", "N/A")}')
print()
print('Files in model directory:')
for fn in sorted(os.listdir(MODEL_DIR)):
    sz = os.path.getsize(os.path.join(MODEL_DIR, fn)) / 1e6
    print(f'  {fn:<40} ({sz:.1f} MB)')
print()
print('Eval outputs saved to /kaggle/working/')
for fn in ['radar_plots.png', 'score_distributions.png', 'training_history.png']:
    p = f'/kaggle/working/{fn}'
    print(f'  {fn}: {"OK" if os.path.exists(p) else "MISSING"}')